# Traitement des données pollution horaire

**Objectif** : préparer les 3 fichiers pollution (NO2, PM10, PM25) en un dataset unifié,
avec les features horaires pour le modèle.

Étapes :
1. Charger les 3 fichiers pollution — format horaire × segment du périph
2. Passer du format *wide* (1 colonne par segment) au format *long* (1 ligne par heure × segment)
3. Fusionner les 3 polluants sur `(time, segment)`
4. Ajouter les features horaires (`HEURE`, `HEURE_SIN`, `HEURE_COS`, `PERIODE`)
5. Sauvegarder

**Résultat attendu** : 70 272 lignes (8784 h × 8 segments) × 10 colonnes.

> La fusion avec `dataset_journalier_2024.csv` (météo + événements + validations) se fera
> dans un notebook séparé.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Notebook situé dans notebooks/, données dans data/processed/
DATA_DIR = Path("../data/processed")
OUT_PATH = DATA_DIR / "pollution_processed_2024.csv"

# Segments du périph dans l'ordre géographique
SEGMENTS = ["Chap-Bagn", "Bagn-Berc", "Berc-Ital", "Ital-A6a",
            "A6a-Sevr", "Sevr-Aute", "Aute-Mail", "Mail-Chap"]

## Étape 1 — Chargement des fichiers pollution

Format actuel : **wide** — chaque colonne est un segment, chaque ligne est une heure.

In [ ]:
no2  = pd.read_csv(DATA_DIR / "2024_NO2_cleaned.csv")
pm10 = pd.read_csv(DATA_DIR / "2024_PM10_cleaned.csv")
pm25 = pd.read_csv(DATA_DIR / "2024_PM25_cleaned.csv")

print(f"NO2  : {no2.shape}")
print(f"PM10 : {pm10.shape}")
print(f"PM25 : {pm25.shape}")
print(f"\nColonnes NO2 : {list(no2.columns)}")
no2.head(3)

## Étape 2 — Passage en format long

Le format *wide* (8 colonnes de segments) n'est pas exploitable directement par XGBoost,
qui prédit une seule cible à la fois. On pivote en format *long* :

- 1 ligne = 1 observation (1 heure × 1 segment)
- Le segment devient une **feature catégorielle**, pas une colonne distincte

On passe de 8784 lignes (heures) à **8784 × 8 = 70 272 lignes**.

In [ ]:
def melt_pollutant(df, name):
    """Pivote wide → long : time + segment + valeur."""
    return df.melt(
        id_vars="time",
        value_vars=SEGMENTS,
        var_name="segment",
        value_name=name,
    )

no2_long  = melt_pollutant(no2,  "NO2")
pm10_long = melt_pollutant(pm10, "PM10")
pm25_long = melt_pollutant(pm25, "PM25")

print(f"NO2  long : {no2_long.shape}")
no2_long.head(10)

## Étape 3 — Fusion des 3 polluants

Les 3 polluants partagent la même clé `(time, segment)`. On les rassemble en une seule table.

Le `validate="one_to_one"` garantit l'absence de doublon — si jamais c'était le cas, le code
planterait au lieu de produire un résultat silencieusement faux.

In [ ]:
pollution = (
    no2_long
    .merge(pm10_long, on=["time", "segment"], validate="one_to_one")
    .merge(pm25_long, on=["time", "segment"], validate="one_to_one")
)

print(f"Table pollution unifiée : {pollution.shape}")
print(f"\nNaN par polluant :")
print(pollution[["NO2", "PM10", "PM25"]].isna().sum())
pollution.head()

## Étape 4 — Features horaires

La pollution suit un cycle journalier marqué (pic matin ~6-8h, plateau, pic soir ~17-19h).
On donne au modèle les outils pour le capter :

- **`HEURE`** (0-23) : heure brute.
- **`HEURE_SIN` / `HEURE_COS`** : encodage cyclique. Sans ça, le modèle voit 23h et 0h comme
  très éloignés alors que ce sont deux heures consécutives. La projection sur un cercle
  rapproche les heures voisines.
- **`PERIODE`** : catégorisation (nuit / pointe matin / journée / pointe soir / soirée). Utile
  pour le dashboard.

On extrait aussi **`DATE`** (YYYY-MM-DD) — ce sera la clé de jointure avec le dataset
journalier dans le notebook suivant.

In [ ]:
# Conversion en datetime + extraction
pollution["time"]  = pd.to_datetime(pollution["time"])
pollution["DATE"]  = pollution["time"].dt.strftime("%Y-%m-%d")
pollution["HEURE"] = pollution["time"].dt.hour

# Encodage cyclique
pollution["HEURE_SIN"] = np.sin(2 * np.pi * pollution["HEURE"] / 24)
pollution["HEURE_COS"] = np.cos(2 * np.pi * pollution["HEURE"] / 24)

# Période catégorielle
def periode(h):
    if   h <  6:  return "nuit"
    elif h < 10:  return "pointe_matin"
    elif h < 17:  return "journee"
    elif h < 21:  return "pointe_soir"
    else:         return "soiree"

pollution["PERIODE"] = pollution["HEURE"].map(periode)

print("Répartition des périodes :")
print(pollution["PERIODE"].value_counts())
pollution[["time", "DATE", "segment", "NO2", "HEURE", "HEURE_SIN", "HEURE_COS", "PERIODE"]].head()

## Sanity checks visuels

Avant de sauvegarder, on vérifie que le pipeline n'a pas corrompu les patterns attendus.

### Profil horaire moyen

On attend la signature classique d'un axe de trafic :
- creux nocturne (1h-4h)
- pic matin (6h-9h)
- plateau journée
- pic soir (17h-20h)

In [ ]:
profil_h = pollution.groupby("HEURE")[["NO2", "PM10", "PM25"]].mean()

fig, ax = plt.subplots(figsize=(11, 4))
profil_h.plot(ax=ax, marker="o")
ax.set_xlabel("Heure")
ax.set_ylabel("Concentration moyenne (µg/m³)")
ax.set_title("Profil horaire moyen — pollution sur le périphérique parisien (2024)")
ax.set_xticks(range(0, 24))
ax.grid(alpha=0.3)
ax.legend(title="Polluant")
plt.tight_layout()
plt.show()

### NO2 par segment du périph

Disparités spatiales : certains segments sont plus chargés que d'autres.

In [ ]:
profil_s = pollution.groupby("segment")["NO2"].mean().reindex(SEGMENTS)

fig, ax = plt.subplots(figsize=(10, 4))
profil_s.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Segment")
ax.set_ylabel("NO2 moyen (µg/m³)")
ax.set_title("NO2 moyen par segment du périphérique (2024)")
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Étape 5 — Sauvegarde

On range les colonnes pour la lisibilité puis on écrit le fichier.

In [ ]:
ordre = ["time", "DATE", "segment",
         "NO2", "PM10", "PM25",
         "HEURE", "HEURE_SIN", "HEURE_COS", "PERIODE"]
pollution = pollution[ordre]

pollution.to_csv(OUT_PATH, index=False)
print(f"✓ Sauvegardé : {OUT_PATH}")
print(f"  {pollution.shape[0]:,} lignes × {pollution.shape[1]} colonnes")
print(f"\nColonnes :")
for c in pollution.columns:
    print(f"  - {c}")

---

**Sortie** : `data/processed/pollution_processed_2024.csv`

Prochaine étape (notebook séparé) : fusion avec `dataset_journalier_2024.csv` sur la clé `DATE`
pour construire le dataset d'entraînement complet.